# Automatic Test Generator

This notebook runs a full loop: an AI **generates** medium-complexity Python
functions (or you **paste** your own), a second set of AI calls **writes pytest
tests** from the intended spec *only* (it never sees the implementation), the
tests are **executed** in an isolated temp dir, and any failures get **one
repair pass** where the model decides whether the code or the test was wrong.
The result is a Markdown **report**.

**Input modes**
- **Mode A** — no code on hand: the model invents N functions. A *buggy* toggle
  makes it plant one subtle bug per function so the repair pass has something to do.
- **Mode B** — paste a Python file: the model pulls out the top-level functions
  and infers a spec for each.

**Models** — Gemini 3.5 Flash Lite (Gemini API) or GPT-OSS 120B (Groq).

## Setup

Imports. Also make sure `uv add pytest pytest-json-report pytest-timeout` has been run.

In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

## API keys and clients

Loads `.env`, builds the two OpenAI-compatible clients, and registers them in `MODELS`.

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

print("Google API Key:", "set" if google_api_key else "NOT set")
print("Groq API Key:  ", "set" if groq_api_key else "NOT set")

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)

MODELS = {
    "Gemini 3.5 Flash Lite": (gemini, "gemini-3.5-flash-lite", "GOOGLE_API_KEY", google_api_key),
    "GPT-OSS 120B (Groq)":   (groq,   "openai/gpt-oss-120b",          "GROQ_API_KEY",   groq_api_key),
}

## The data model

`FunctionCase` is carried through every stage. `RunResult` holds one pytest run's outcome.


In [ ]:
@dataclass
class RunResult:
    passed: int = 0
    failed: int = 0
    errors: int = 0
    tracebacks: list = field(default_factory=list)
    raw_json: dict = field(default_factory=dict)

    @property
    def green(self) -> bool:
        return self.failed == 0 and self.errors == 0 and self.passed > 0


@dataclass
class FunctionCase:
    name: str
    spec: str
    code: str
    planted_bug: str | None = None
    tests: str = ""
    round1: RunResult | None = None
    repair_note: str | None = None
    repaired_code: str | None = None
    repaired_tests: str | None = None
    round2: RunResult | None = None
    error: str | None = None


## LLM call helper

Thin wrapper over `client.chat.completions.create`. `chat_json` forces JSON mode and retries once on bad JSON.


In [ ]:
def chat(client, model, system, user, json_mode=False):
    kwargs = {}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        **kwargs,
    )
    text = resp.choices[0].message.content.strip()
    if not json_mode and text.startswith("```"):
        parts = text.split("```")
        text = parts[1] if len(parts) > 1 else text
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return text



def chat_json(client, model, system, user):
    if "json" not in (system + user).lower():
        user = user + "\n\nRespond with JSON only."
    raw = chat(client, model, system, user, json_mode=True)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        raw = chat(client, model, system, user + "\n\nReturn valid JSON only.", json_mode=True)
        return json.loads(raw)



## The runner

Writes `solution.py` + `test_solution.py` to a temp dir and runs pytest with a JSON report and a 10s per-test timeout. Parses the report into a `RunResult`. Temp dir always cleaned up.


In [ ]:
def run_case(code: str, tests: str) -> RunResult:
    tmp = tempfile.mkdtemp(prefix="testgen_")
    try:
        Path(tmp, "solution.py").write_text(code, encoding="utf-8")
        Path(tmp, "test_solution.py").write_text(tests, encoding="utf-8")
        Path(tmp, "conftest.py").write_text("", encoding="utf-8")

        try:
            proc = subprocess.run(
                [sys.executable, "-m", "pytest", "-q",
                 "--json-report", "--json-report-file=report.json",
                 "--timeout=10"],
                cwd=tmp, capture_output=True, text=True, timeout=60,
            )
        except subprocess.TimeoutExpired:
            return RunResult(errors=1, tracebacks=["subprocess timeout (60s)"])

        report_path = Path(tmp, "report.json")
        if not report_path.exists():
            return RunResult(errors=1, tracebacks=[(proc.stdout + proc.stderr).strip()])

        data = json.loads(report_path.read_text(encoding="utf-8"))
        passed = failed = errors = 0
        tracebacks = []
        for t in data.get("tests", []):
            outcome = t.get("outcome")
            longrepr = (
                t.get("call", {}).get("longrepr")
                or t.get("setup", {}).get("longrepr")
                or ""
            )
            if outcome == "passed":
                passed += 1
            elif outcome == "failed":
                failed += 1
                tracebacks.append(str(longrepr))
            else:
                errors += 1
                tracebacks.append(str(longrepr) or str(outcome))
        return RunResult(passed, failed, errors, tracebacks, data)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


## The report

Header with run config and totals, a summary table, then a collapsible block per function with spec, planted bug, code, tests, round-1 results, the repair, and round-2 results.


In [ ]:
def _trim(tb: str, n: int = 15) -> str:
    lines = str(tb).splitlines()
    return "\n".join(lines[-n:])


def _culprit(c) -> str:
    if c.repaired_code:
        return "code"
    if c.repaired_tests:
        return "test"
    return "-"


def build_report(cases: list, config: dict) -> str:
    total = len(cases)
    green = sum(1 for c in cases
               if (c.round2 or c.round1) and (c.round2 or c.round1).green)

    out = [
        "# Test Generator Report",
        "",
        f"- **Mode:** {config.get('mode')}",
        f"- **Model:** {config.get('model')}",
        f"- **Buggy:** {config.get('buggy')}",
        f"- **Time:** {datetime.now():%Y-%m-%d %H:%M:%S}",
        f"- **Result:** {green}/{total} functions green after repair",
        "",
        "| function | round 1 (P/F/E) | repaired? | culprit | round 2 (P/F/E) | final |",
        "|---|---|---|---|---|---|",
    ]
    for c in cases:
        r1 = c.round1
        r1s = f"{r1.passed}/{r1.failed}/{r1.errors}" if r1 else "-"
        r2 = c.round2
        r2s = f"{r2.passed}/{r2.failed}/{r2.errors}" if r2 else "-"
        repaired = "yes" if c.repair_note else "no"
        if c.error:
            final = "error"
        else:
            last = c.round2 or c.round1
            final = ("green" if last.green else "red") if last else "n/a"
        out.append(f"| {c.name} | {r1s} | {repaired} | {_culprit(c)} | {r2s} | {final} |")
    out.append("")

    for c in cases:
        out.append(f"<details><summary><b>{c.name}</b></summary>")
        out.append("")
        if c.error:
            out += [f"**Error:** {c.error}", "", "</details>", ""]
            continue
        out += [f"**Spec:** {c.spec}", ""]
        if c.planted_bug:
            out += [f"**Planted bug:** {c.planted_bug}", ""]
        out += ["**Code (round 1):**", "```python", c.code, "```"]
        out += ["**Tests:**", "```python", c.tests, "```"]
        if c.round1:
            out.append(f"**Round 1:** {c.round1.passed} passed, "
                       f"{c.round1.failed} failed, {c.round1.errors} errors")
            for tb in c.round1.tracebacks:
                out += ["```", _trim(tb), "```"]
        if c.repair_note:
            out += [f"**Repair — culprit: {_culprit(c)}**", c.repair_note, ""]
            if c.repaired_code:
                out += ["**Fixed code:**", "```python", c.repaired_code, "```"]
            if c.repaired_tests:
                out += ["**Fixed tests:**", "```python", c.repaired_tests, "```"]
        if c.round2:
            out.append(f"**Round 2:** {c.round2.passed} passed, "
                       f"{c.round2.failed} failed, {c.round2.errors} errors")
            for tb in c.round2.tracebacks:
                out += ["```", _trim(tb), "```"]
        out += ["", "</details>", ""]

    return "\n".join(out)


## Report check

Feed `build_report` a hand-built list covering: clean pass, repaired-code, and errored case. Eyeball the Markdown.


In [ ]:
_c1 = FunctionCase(name="clean", spec="adds", code="def f(): ...", tests="def test(): ...")
_c1.round1 = RunResult(passed=3)

_c2 = FunctionCase(name="fixed", spec="adds", code="bug", tests="t")
_c2.round1 = RunResult(passed=1, failed=2, tracebacks=["E   assert 1 == 2\n  more\n  lines"])
_c2.repair_note = "Off-by-one in the loop bound."
_c2.repaired_code = "def f(): return 1"
_c2.round2 = RunResult(passed=3)

_c3 = FunctionCase(name="broke", spec="x", code="y", tests="z", error="boom")

_md = build_report([_c1, _c2, _c3], {"mode": "A", "model": "Gemini 3.5 Flash Lite", "buggy": True})
display(Markdown(_md))
assert "2/3 functions green after repair" in _md
assert "<details><summary><b>fixed</b></summary>" in _md
print("report OK")


## Prompts

Four jobs: invent sample functions (Mode A), parse pasted code (Mode B), write tests from a spec, repair a failing function. All ask for strict JSON.


In [ ]:
SAMPLE_SYSTEM = (
    "You write medium-complexity Python functions for a testing exercise. "
    "Not trivial (no one-liners), not sprawling. Each function should be mostly pure, "
    "deterministic, standard library only, take 1-3 arguments, and have a clear return "
    "value. Draw from: string and list manipulation, small algorithms, date math, "
    "parsing, number theory."
)

def sample_user(n: int, buggy: bool) -> str:
    u = (
        f"Generate {n} functions. For each provide:\n"
        "- `name`\n"
        "- `spec`: 2-4 sentences describing intended behaviour, arguments, return value, "
        "and edge cases. Describe behaviour only, never reference an implementation.\n"
        "- `code`: an implementation with signature and docstring.\n"
    )
    if buggy:
        u += (
            "\nIntroduce exactly ONE subtle bug per function - an off-by-one, a wrong "
            "boundary, a swapped operator, mishandled empty input, or a wrong default. "
            "The code must still run (no syntax errors). The `spec` must describe the "
            "CORRECT intended behaviour.\n"
            "Also provide `planted_bug`: one sentence naming the bug.\n"
            'Return {"functions": [{"name": ..., "spec": ..., "code": ..., "planted_bug": ...}]}.'
        )
    else:
        u += '\nReturn {"functions": [{"name": ..., "spec": ..., "code": ...}]}.'
    return u


PARSE_SYSTEM = (
    "You extract testable units from Python source and describe their intended behaviour."
)

def parse_user(src: str) -> str:
    return (
        "Here is Python source. For each top-level function (skip private `_name` "
        "functions, skip `main` and CLI glue) provide:\n"
        "- `name`\n"
        "- `code`: the verbatim source of that function, plus any module-level helper or "
        "constant it needs to run\n"
        "- `spec`: 2-4 sentences inferred from the code, its docstring, and its name\n\n"
        'Return {"functions": [{"name": ..., "spec": ..., "code": ...}]}.\n\n'
        f"```python\n{src}\n```"
    )


## build specs

Mode A asks the model to invent functions; Mode B asks it to parse the pasted source. Both return a list of `FunctionCase` with `name`, `spec`, `code` (and `planted_bug` in buggy Mode A).


In [ ]:
def build_specs(config: dict) -> list:
    client, model, _, _ = MODELS[config["model"]]
    if config["mode"] == "A":
        data = chat_json(client, model, SAMPLE_SYSTEM,
                         sample_user(config["n_functions"], config["buggy"]))
    else:
        data = chat_json(client, model, PARSE_SYSTEM, parse_user(config["source_code"]))

    cases = []
    for f in data.get("functions", []):
        cases.append(FunctionCase(
            name=f["name"],
            spec=f["spec"],
            code=f["code"],
            planted_bug=f.get("planted_bug"),
        ))
    return cases


## generate tests

One call per function. The model is given the name and spec **only** — never the implementation — so buggy code genuinely fails.


In [ ]:
TEST_SYSTEM = (
    "You write thorough pytest tests from a specification. You do NOT see the "
    "implementation. Cover typical cases, boundaries, empty or zero inputs, and invalid "
    "input if the spec implies it. Write 4-8 tests as plain pytest functions, no classes. "
    "Assume `from solution import <name>`. Reply with ONLY a Python code block, nothing else."
)

def _extract_code(text: str) -> str:
    text = text.strip()
    if "```" in text:
        block = text.split("```", 2)[1]
        block = block.split("```")[0]
    else:
        block = text
    lines = block.splitlines()
    if lines and lines[0].strip().lower() in ("python", "py"):
        lines = lines[1:]
    return "\n".join(lines).strip()


def generate_tests(case, client, model):
    user = (
        f"Function name: {case.name}\n"
        f"Specification:\n{case.spec}\n\n"
        "Return a single Python code block with the full pytest module."
    )
    raw = chat(client, model, TEST_SYSTEM, user)
    code = _extract_code(raw)
    try:
        compile(code, "<tests>", "exec")
    except SyntaxError:
        raw = chat(client, model, TEST_SYSTEM,
                   user + "\n\nYour previous output had a syntax error. Return valid Python only.")
        code = _extract_code(raw)
    case.tests = code
    return case


## repair pass

Only runs for functions with failures/errors. The model gets code + tests + tracebacks and returns which side to fix. Exactly one re-run, whatever the outcome.


In [ ]:
REPAIR_SYSTEM = (
    "A function failed its tests. Decide whether the bug is in the CODE or in the TEST, "
    "fix that one only, and explain briefly. Reply with ONLY a JSON object."
)

def repair(case, client, model):
    tb = "\n\n".join(case.round1.tracebacks) if case.round1 else ""
    user = (
        f"Specification:\n{case.spec}\n\n"
        f"Current code:\n{case.code}\n\n"
        f"Current tests:\n{case.tests}\n\n"
        f"pytest output:\n{tb}\n\n"
        'Return {"culprit": "code" or "test", "fixed_code": full code string or null, '
        '"fixed_tests": full test module string or null, "note": "<= 2 sentences"}.'
    )
    data = chat_json(client, model, REPAIR_SYSTEM, user)
    case.repair_note = data.get("note", "")

    code, tests = case.code, case.tests
    fixed_code = data.get("fixed_code")
    fixed_tests = data.get("fixed_tests")
    if data.get("culprit") == "code" and fixed_code:
        case.repaired_code, code = fixed_code, fixed_code
    elif data.get("culprit") == "test" and fixed_tests:
        case.repaired_tests, tests = fixed_tests, fixed_tests
    elif fixed_code:
        case.repaired_code, code = fixed_code, fixed_code
    elif fixed_tests:
        case.repaired_tests, tests = fixed_tests, fixed_tests

    case.round2 = run_case(code, tests)
    return case


## Orchestration

Ties the stages together: guard inputs, build specs, then per function generate tests, run, and repair if needed. Per-function exceptions are caught and shown in the report; the run continues.


In [ ]:
def run_pipeline(mode, model_label, n_functions, buggy, source_code):
    config = {
        "mode": mode,
        "model": model_label,
        "n_functions": int(n_functions),
        "buggy": bool(buggy),
        "source_code": source_code or "",
    }
    client, model, key_name, key_val = MODELS[model_label]
    if not key_val:
        return f"**{key_name} is not set.** Add it to your `.env` and restart the kernel."
    if mode == "B" and not config["source_code"].strip():
        return "**Mode B selected but no source code was pasted.**"

    try:
        cases = build_specs(config)
    except Exception as e:
        return f"**Failed to build specs:** {e}"
    if not cases:
        return "**No functions found.**"

    for c in cases:
        try:
            generate_tests(c, client, model)
            c.round1 = run_case(c.code, c.tests)
            if c.round1.failed + c.round1.errors > 0:
                repair(c, client, model)
        except Exception as e:
            c.error = str(e)

    return build_report(cases, config)


## Gradio app

Mode radio, model dropdown, function-count slider, buggy checkbox, source textbox, Run button, Markdown output. `n_functions` and `buggy` are ignored in Mode B; `source_code` is ignored in Mode A.


In [ ]:
def run_pipeline_ui(mode, model_label, n_functions, buggy, source_code):
    # first yield paints immediately when the button is clicked
    yield "⏳ **Generating…**  building functions, writing tests, running pytest — this can take a minute."
    yield run_pipeline(mode, model_label, n_functions, buggy, source_code)


with gr.Blocks(title="Automatic Test Generator") as demo:
    gr.Markdown(
        "# Automatic Test Generator\n"
        "Generate or paste Python functions, auto-write pytest tests from the spec, "
        "run them, repair once, and report."
    )
    with gr.Row():
        mode = gr.Radio(["A", "B"], value="A",
                        label="Mode  (A = generate functions, B = paste your own)")
        model_label = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Model")
    with gr.Row():
        n_functions = gr.Slider(1, 6, value=3, step=1, label="How many functions (Mode A)")
        buggy = gr.Checkbox(value=False, label="Generate buggy functions (Mode A)")
    source_code = gr.Textbox(lines=12, label="Paste Python source (Mode B)",
                             placeholder="def foo(x):\n    ...")
    run_btn = gr.Button("Run", variant="primary")
    output = gr.Markdown()

    run_btn.click(
        lambda: gr.update(value="Running…", interactive=False),
        outputs=run_btn,
    ).then(
        run_pipeline_ui,
        inputs=[mode, model_label, n_functions, buggy, source_code],
        outputs=output,
    ).then(
        lambda: gr.update(value="Run", interactive=True),
        outputs=run_btn,
    )

demo.launch()
